### Import Libraries

In [1]:
# -*- coding: utf-8 -*-

import os
import sys
import time
import math
import copy
import random
import numpy as np
from collections import deque
from typing import List, Tuple, Dict, Any, Optional
import warnings
warnings.filterwarnings('ignore')

### Game Entities

In [2]:
def create_car(x, y, color=(255, 255, 255), is_player=False, car_id=0):
    """Create car"""
    return {
        'x': x,
        'y': y,
        'color': color,
        'is_player': is_player,
        'car_id': car_id,
        'speed': 3.0,
        'direction': 0,
        'acceleration': 0.2,
        'turn_speed': 3,
        'max_speed': 5,
        'min_speed': 1,
        'health': 100,
        'has_shield': False,
        'shield_time': 0,
        'boost_time': 0,
        'lap_progress': 0,
        'laps_completed': 0,
        'prev_angle': None,
        'cumulative_angle': 0.0,
        'distance_traveled': 0,
        'collision_cooldown': 0,
        'collected_power_up': False,
        'near_start_line': False,
        'lap_completed_this_step': False,
        'is_going_wrong_way': False,
        'hit_wall_this_step': False,
        'hit_car_this_step': False,
        'was_hit_this_step': False,
        'lap_rank': 0,
        'width': 24,
        'height': 16
    }

def create_obstacle(x, y, size=30):
    """Create obstacle"""
    return {
        'x': x,
        'y': y,
        'size': size,
        'color': (150, 150, 150)
    }

def create_power_up(x, y, power_up_type='bonus'):
    """Create power-up - Single power-up type with small bonus"""
    return {
        'x': x,
        'y': y,
        'power_up_type': power_up_type,
        'color': (255, 215, 0),  # 金色道具
        'size': 15
    }

def create_track(width, height, track_width=400, track_height=300, track_type='oval'):
    """Create track - 支持两种不规则形状"""
    center_x = width // 2
    center_y = height // 2
    track = []

    if track_type == 'oval':
        # 椭圆形赛道（原始）
        for angle in np.linspace(0, 2 * math.pi, 100):
            x = center_x + (track_width // 2) * math.cos(angle)
            y = center_y + (track_height // 2) * math.sin(angle)
            track.append((int(x), int(y)))
    elif track_type == 'peanut':
        # 花生形赛道（8字形变体）
        for angle in np.linspace(0, 2 * math.pi, 100):
            r_factor = 1 + 0.3 * math.sin(2 * angle)  # 花生形状
            x = center_x + (track_width // 2) * r_factor * math.cos(angle)
            y = center_y + (track_height // 2) * r_factor * math.sin(angle)
            track.append((int(x), int(y)))
    elif track_type == 'kidney':
        # 肾形赛道（不对称）
        for angle in np.linspace(0, 2 * math.pi, 100):
            r_factor = 1 + 0.2 * math.sin(angle) + 0.15 * math.cos(3 * angle)
            x = center_x + (track_width // 2) * r_factor * math.cos(angle)
            y = center_y + (track_height // 2) * r_factor * math.sin(angle)
            track.append((int(x), int(y)))

    return track, track_type

def is_within_track(x, y, center_x, center_y, track_width, track_height, track_type='oval'):
    """Check if point is within track（外椭圆和内椭圆之间）"""
    dx = x - center_x
    dy = y - center_y
    angle = math.atan2(dy, dx)

    inner_ratio = 0.5

    if track_type == 'oval':
        # 椭圆形
        value = (dx ** 2) / ((track_width / 2) ** 2) + (dy ** 2) / ((track_height / 2) ** 2)
        inner_value = (dx ** 2) / ((track_width * inner_ratio / 2) ** 2) + \
                      (dy ** 2) / ((track_height * inner_ratio / 2) ** 2)
    elif track_type == 'peanut':
        # 花生形 - 修正计算方法
        r_factor = 1 + 0.3 * math.sin(2 * angle)
        # 外圈半径（考虑椭圆和r_factor）
        outer_r = math.sqrt(((track_width / 2) * math.cos(angle)) ** 2 + \
                           ((track_height / 2) * math.sin(angle)) ** 2) * r_factor
        # 内圈半径
        inner_r = math.sqrt(((track_width * inner_ratio / 2) * math.cos(angle)) ** 2 + \
                           ((track_height * inner_ratio / 2) * math.sin(angle)) ** 2) * r_factor
        dist = math.sqrt(dx ** 2 + dy ** 2)
        value = (dist / outer_r) ** 2
        inner_value = (dist / inner_r) ** 2
    elif track_type == 'kidney':
        # 肾形 - 修正计算方法
        r_factor = 1 + 0.2 * math.sin(angle) + 0.15 * math.cos(3 * angle)
        # 外圈半径（考虑椭圆和r_factor）
        outer_r = math.sqrt(((track_width / 2) * math.cos(angle)) ** 2 + \
                           ((track_height / 2) * math.sin(angle)) ** 2) * r_factor
        # 内圈半径
        inner_r = math.sqrt(((track_width * inner_ratio / 2) * math.cos(angle)) ** 2 + \
                           ((track_height * inner_ratio / 2) * math.sin(angle)) ** 2) * r_factor
        dist = math.sqrt(dx ** 2 + dy ** 2)
        value = (dist / outer_r) ** 2
        inner_value = (dist / inner_r) ** 2
    else:
        # 默认椭圆
        value = (dx ** 2) / ((track_width / 2) ** 2) + (dy ** 2) / ((track_height / 2) ** 2)
        inner_value = (dx ** 2) / ((track_width * inner_ratio / 2) ** 2) + \
                      (dy ** 2) / ((track_height * inner_ratio / 2) ** 2)

    return value <= 1.0 and inner_value >= 1.0

def update_lap_progress(car, center_x, center_y):
    """Update lap progress，基于绕中心的累积角度"""
    dx = car['x'] - center_x
    dy = car['y'] - center_y
    current_angle = math.atan2(dy, dx)

    if car['prev_angle'] is not None:
        delta = current_angle - car['prev_angle']
        # 处理角度跨越 -pi/pi 边界
        if delta > math.pi:
            delta -= 2 * math.pi
        elif delta < -math.pi:
            delta += 2 * math.pi

        # Pygame坐标系（y轴向下），atan2(dy, dx):
        # 右(0°) → 下(90°) → 左(±180°) → 上(-90°) → 右(0°)
        # 顺时针：delta > 0（大部分时候），但跨越±180°时delta < 0
        # 简化：直接累积delta，完整一圈累积约2π（考虑边界跳变）
        car['cumulative_angle'] += delta

        # 顺时针完成一圈：累积角度 >= 2π
        if car['cumulative_angle'] >= 2 * math.pi:
            car['laps_completed'] += 1
            car['cumulative_angle'] -= 2 * math.pi  # 保留余数
            car['lap_completed_this_step'] = True
        else:
            car['lap_completed_this_step'] = False

        # 逆行检测：累积角度 < -π/2（明显逆时针）
        car['is_going_wrong_way'] = car['cumulative_angle'] < -math.pi / 2

    car['prev_angle'] = current_angle
    # lap_progress: 0~1 表示当前圈的进度
    car['lap_progress'] = max(0.0, min(1.0, car['cumulative_angle'] / (2 * math.pi)))
    return car

def check_collision(obj1, obj2):
    """Check collision between two objects"""
    if 'size' in obj2:  # 障碍物
        distance = math.sqrt((obj1['x'] - obj2['x'])**2 + (obj1['y'] - obj2['y'])**2)
        return distance < (obj1['width'] / 2 + obj2['size'])
    elif 'power_up_type' in obj2:  # 道具
        distance = math.sqrt((obj1['x'] - obj2['x'])**2 + (obj1['y'] - obj2['y'])**2)
        return distance < (obj1['width'] / 2 + 15)
    else:  # 与其他赛车碰撞
        return (abs(obj1['x'] - obj2['x']) < (obj1['width'] + obj2['width']) / 2 and
                abs(obj1['y'] - obj2['y']) < (obj1['height'] + obj2['height']) / 2)

def update_car(car, action, track_params, obstacles, power_ups):
    """Update car state"""
    center_x, center_y, track_width, track_height, track_type = track_params

    # 重置撞墙标记
    car['hit_wall_this_step'] = False

    # 首先检查赛车当前位置是否在赛道外（可能由于之前的bug导致）
    if not is_within_track(car['x'], car['y'], center_x, center_y, track_width, track_height, track_type):
        # 立即将赛车弹回赛道中线并扣血
        angle_to_center = math.atan2(car['y'] - center_y, car['x'] - center_x)
        # 计算该角度的r_factor（考虑赛道形状）
        if track_type == 'peanut':
            r_factor = 1 + 0.3 * math.sin(2 * angle_to_center)
        elif track_type == 'kidney':
            r_factor = 1 + 0.2 * math.sin(angle_to_center) + 0.15 * math.cos(3 * angle_to_center)
        else:
            r_factor = 1.0
        mid_ratio = 0.75
        car['x'] = center_x + (track_width * mid_ratio / 2) * r_factor * math.cos(angle_to_center)
        car['y'] = center_y + (track_height * mid_ratio / 2) * r_factor * math.sin(angle_to_center)
        car['speed'] = car['min_speed']
        car['direction'] = math.degrees(angle_to_center + math.pi / 2)
        # 撞墙扣血
        if car['collision_cooldown'] == 0:
            car['health'] -= 15
            car['collision_cooldown'] = 30
            car['hit_wall_this_step'] = True

    # 处理Action (0: No-op, 1: Brake, 2: Left, 3: Right)
    if action == 1:  # Brake
        car['speed'] = max(car['speed'] - car['acceleration'] * 1.5, car['min_speed'])
    elif action == 2:  # Left
        car['direction'] = (car['direction'] - car['turn_speed']) % 360
    elif action == 3:  # Right
        car['direction'] = (car['direction'] + car['turn_speed']) % 360

    # 更新位置
    dx = car['speed'] * math.cos(math.radians(car['direction']))
    dy = car['speed'] * math.sin(math.radians(car['direction']))

    new_x = car['x'] + dx
    new_y = car['y'] + dy

    # 检查新位置是否在赛道内
    if is_within_track(new_x, new_y, center_x, center_y, track_width, track_height, track_type):
        car['x'] = new_x
        car['y'] = new_y
        car['distance_traveled'] += math.sqrt(dx*dx + dy*dy)
        # Update lap progress
        update_lap_progress(car, center_x, center_y)
    else:
        # 碰到墙壁，小幅度弹回到赛道内并扣血
        angle_to_center = math.atan2(car['y'] - center_y, car['x'] - center_x)

        # 计算该角度的r_factor（考虑赛道形状）
        if track_type == 'peanut':
            r_factor = 1 + 0.3 * math.sin(2 * angle_to_center)
        elif track_type == 'kidney':
            r_factor = 1 + 0.2 * math.sin(angle_to_center) + 0.15 * math.cos(3 * angle_to_center)
        else:
            r_factor = 1.0
        
        # 计算当前距离中心的归一化距离
        dx = car['x'] - center_x
        dy = car['y'] - center_y
        current_dist = math.sqrt((dx / (track_width / 2)) ** 2 + (dy / (track_height / 2)) ** 2)

        # 小幅度弹回：如果在外墙外，弹回到0.95；如果在内墙内，弹回到0.55
        if current_dist > 1.0:  # 撞外墙
            bounce_ratio = 0.95
        else:  # 撞内墙
            bounce_ratio = 0.55

        # 弹回位置考虑r_factor
        bounce_x = center_x + (track_width * bounce_ratio / 2) * r_factor * math.cos(angle_to_center)
        bounce_y = center_y + (track_height * bounce_ratio / 2) * r_factor * math.sin(angle_to_center)

        # 更新赛车位置到弹回位置
        car['x'] = bounce_x
        car['y'] = bounce_y

        # 速度降低作为碰墙惩罚
        car['speed'] *= 0.5
        car['speed'] = max(car['speed'], car['min_speed'])

        # 调整方向沿着赛道切线方向（顺时针）
        car['direction'] = math.degrees(angle_to_center + math.pi / 2)

        # 撞墙扣血（确保每次撞墙都扣血）
        if car['collision_cooldown'] == 0:
            car['health'] -= 15
            car['collision_cooldown'] = 30
            car['hit_wall_this_step'] = True

    # 更新状态

    # 更新特殊效果
    if car['shield_time'] > 0:
        car['shield_time'] -= 1
        if car['shield_time'] <= 0:
            car['has_shield'] = False

    if car['boost_time'] > 0:
        car['boost_time'] -= 1

    # 更新碰撞冷却
    if car['collision_cooldown'] > 0:
        car['collision_cooldown'] -= 1

    return car

def handle_collision(car):
    """Handle collision"""
    if car['collision_cooldown'] > 0:
        return car  # 冷却期内不重复扣血
    if car['has_shield']:
        car['has_shield'] = False
        car['shield_time'] = 0
    else:
        car['health'] -= 10
        car['speed'] *= 0.7
        car['collision_cooldown'] = 30

    car['health'] = max(car['health'], 0)
    return car

def collect_power_up(car, power_up):
    """Collect power-up - Only provides small bonus, no effects"""
    car['collected_power_up'] = True
    return car

# @title 2.2 游戏状态管理函数

def create_game_state(width=800, height=600, num_ai_cars=3, track_type='oval'):
    """Create game state"""
    state = {
        'width': width,
        'height': height,
        'num_ai_cars': num_ai_cars,
        'track_center_x': width // 2,
        'track_center_y': height // 2,
        'track_width': 650,
        'track_height': 490,
        'track_type': track_type,
        'max_speed': 8,
        'min_speed': 1,
        'fps': 120,
        'laps': 0,
        'max_laps': 3,
        'is_game_over': False,
        'winner': None,
        'time': 0
    }

    # Create track（拓宽外圈）
    state['track'] = create_track(width, height, 650, 490, track_type=track_type)

    # 赛道中线半径比例 (内圈50%~外圈100%的中间)
    mid_ratio = 0.75

    # 起跑线位于52.5°角度处（45°-60°扇形区域中心）
    start_angle = math.radians(52.5)
    mid_radius_x = state['track_width'] * mid_ratio / 2
    mid_radius_y = state['track_height'] * mid_ratio / 2

    # 起跑线中心点
    start_center_x = state['track_center_x'] + mid_radius_x * math.cos(start_angle)
    start_center_y = state['track_center_y'] + mid_radius_y * math.sin(start_angle)

    # 起跑线方向（与径向呈30°夹角）
    line_angle = start_angle + math.radians(30)

    # 赛道宽度
    outer_r = state['track_height'] / 2
    inner_r = state['track_height'] * 0.5 / 2
    track_band = outer_r - inner_r
    car_spacing = track_band / (num_ai_cars + 2)

    # 所有赛车沿起跑线排列（垂直于起跑线方向）
    perp_angle = line_angle + math.pi / 2  # 垂直于起跑线
    player_offset = -track_band / 2 + car_spacing
    player_x = start_center_x + player_offset * math.cos(perp_angle)
    player_y = start_center_y + player_offset * math.sin(perp_angle)

    ai_positions = []
    for i in range(num_ai_cars):
        offset = -track_band / 2 + car_spacing * (i + 2)
        ai_x = start_center_x + offset * math.cos(perp_angle)
        ai_y = start_center_y + offset * math.sin(perp_angle)
        ai_positions.append((int(ai_x), int(ai_y)))

    state['start_positions'] = {
        'player': (int(player_x), int(player_y)),
        'ai': ai_positions
    }

    # 赛车初始方向：沿赛道切线方向（顺时针）
    # 在52.5°位置，顺时针切线方向约为52.5° + 90° = 142.5°
    initial_direction = math.degrees(start_angle + math.pi / 2)
    state['initial_direction'] = initial_direction  # 存储初始方向供复活使用

    # 创建玩家赛车
    state['player_car'] = create_car(
        x=int(player_x),
        y=int(player_y),
        color=(0, 255, 0),
        is_player=True
    )
    state['player_car']['direction'] = initial_direction

    # 创建AI Car - 全部在同一起跑线，沿起跑线排列
    state['ai_cars'] = []
    ai_colors = [(255, 50, 50), (255, 150, 0), (200, 0, 255)]

    for i in range(min(num_ai_cars, 3)):
        ai_x, ai_y = ai_positions[i]
        ai_car = create_car(
            x=ai_x,
            y=ai_y,
            color=ai_colors[i % len(ai_colors)],
            is_player=False,
            car_id=i
        )
        ai_car['direction'] = initial_direction  # 沿赛道切线方向
        state['ai_cars'].append(ai_car)

    # Create obstacle - 固定3个，放在赛道中央，避开终点线周围（45°-60°扇形区域）
    state['obstacles'] = []
    start_angle_min = math.radians(45)
    start_angle_max = math.radians(60)

    for _ in range(3):  # 固定3个障碍物
        # 生成角度，避开终点线区域（45°-60°）
        while True:
            angle = random.uniform(0, 2 * math.pi)
            # 检查是否在终点线扇形区域外
            if not (start_angle_min <= angle <= start_angle_max):
                break

        # 距离设置在赛道中央区域（70%-80%半径）
        distance = random.uniform(0.70, 0.80)
        x = state['track_center_x'] + int(state['track_width'] * distance * math.cos(angle) / 2)
        y = state['track_center_y'] + int(state['track_height'] * distance * math.sin(angle) / 2)
        state['obstacles'].append(create_obstacle(x, y, size=random.randint(10, 18)))

    # Create power-up - Single power-up type with small bonus，确保至少有5个
    state['power_ups'] = []
    attempts = 0
    max_attempts = 200
    min_power_ups = 5  # 确保至少有5个道具
    while len(state['power_ups']) < min_power_ups and attempts < max_attempts:
        angle = random.uniform(0, 2 * math.pi)
        distance = random.uniform(0.6, 0.9)
        x = state['track_center_x'] + int(state['track_width'] * distance * math.cos(angle) / 2)
        y = state['track_center_y'] + int(state['track_height'] * distance * math.sin(angle) / 2)
        # 验证道具是否在赛道内
        if is_within_track(x, y, state['track_center_x'], state['track_center_y'],
                          state['track_width'], state['track_height'], state.get('track_type', 'oval')):
            state['power_ups'].append(create_power_up(x, y))
        attempts += 1

    # 如果尝试次数用完仍未达到最小数量，强制添加道具（放宽位置限制）
    while len(state['power_ups']) < min_power_ups:
        angle = random.uniform(0, 2 * math.pi)
        distance = 0.75  # 固定在中线位置
        x = state['track_center_x'] + int(state['track_width'] * distance * math.cos(angle) / 2)
        y = state['track_center_y'] + int(state['track_height'] * distance * math.sin(angle) / 2)
        state['power_ups'].append(create_power_up(x, y))

    return state

def get_state_vector(game_state, car):
    """Get state vector（36维）"""
    state = []

    # 1. 自身状态 (7维)
    state.extend([car['x'] / game_state['width'], car['y'] / game_state['height']])
    state.append(car['speed'] / game_state['max_speed'])
    state.append(car['direction'] / 360)
    state.append(1 if car['has_shield'] else 0)
    state.append(car['lap_progress'])
    state.append(car['laps_completed'] / game_state['max_laps'])

    # 2. 最近障碍物 (10维: 5个障碍物 × 距离和角度)
    # 确保我们有5个障碍物的数据
    obstacle_distances = []
    for obs in game_state['obstacles'][:5]:  # 最多取5个
        dx = obs['x'] - car['x']
        dy = obs['y'] - car['y']
        distance = math.sqrt(dx*dx + dy*dy)
        angle = math.atan2(dy, dx)
        obstacle_distances.append((distance, angle))

    # 按距离排序
    obstacle_distances.sort(key=lambda x: x[0])

    # 填充到5个
    for i in range(5):
        if i < len(obstacle_distances):
            dist, angle = obstacle_distances[i]
            state.append(dist / 1000)  # 归一化
            state.append(angle / (2 * math.pi))  # 归一化到[-1, 1]
        else:
            state.extend([1.0, 0.0])  # 默认值（远距离，0角度）

    # 3. 最近对手 (15维: 5个对手 × 距离、角度、速度)
    other_cars = [c for c in game_state['ai_cars'] + [game_state['player_car']] if c != car]
    car_distances = []

    for other in other_cars[:5]:  # 最多取5个
        dx = other['x'] - car['x']
        dy = other['y'] - car['y']
        distance = math.sqrt(dx*dx + dy*dy)
        angle = math.atan2(dy, dx)
        car_distances.append((distance, angle, other['speed']))

    car_distances.sort(key=lambda x: x[0])

    for i in range(5):
        if i < len(car_distances):
            dist, angle, speed = car_distances[i]
            state.append(dist / 1000)
            state.append(angle / (2 * math.pi))
            state.append(speed / game_state['max_speed'])
        else:
            state.extend([1.0, 0.0, 0.0])  # 默认值

    # 4. 最近道具 (4维: 4个道具 × 距离)
    power_up_distances = []

    for pu in game_state['power_ups'][:4]:  # 最多取4个
        dx = pu['x'] - car['x']
        dy = pu['y'] - car['y']
        distance = math.sqrt(dx*dx + dy*dy)
        power_up_distances.append(distance)

    power_up_distances.sort()

    for i in range(4):
        if i < len(power_up_distances):
            state.append(power_up_distances[i] / 1000)
        else:
            state.append(1.0)  # 默认值（远距离）

    return np.array(state, dtype=np.float32)

def calculate_reward(game_state, car):
    """计算Reward - Optimized version"""
    reward = 0

    # 1. Survival Reward
    reward += 0.3

    # 2. Speed Reward
    reward += car['speed'] / car['max_speed'] * 0.3

    # Steps penalty 
    reward -= 0.2

    # 3. Stay on Track Center Line Reward
    dx = car['x'] - game_state['track_center_x']
    dy = car['y'] - game_state['track_center_y']
    norm_dist = math.sqrt((dx / (game_state['track_width'] / 2)) ** 2 +
                        (dy / (game_state['track_height'] / 2)) ** 2)
    ideal_dist = 0.75
    track_deviation = abs(norm_dist - ideal_dist)
    
    reward += max(0, 1.0 - track_deviation * 2.0)

    # 4. Lap Progress Reward - Encourage forward movement along the track
    reward += car['lap_progress'] * 2.0

    # 4.5 Lap Progress Increment Reward 
    if 'last_lap_progress' not in car:
        car['last_lap_progress'] = 0
    progress_delta = car['lap_progress'] - car['last_lap_progress']
    if progress_delta > 0:  
        reward += progress_delta * 100  # Give 100 points reward per 1% progress
    car['last_lap_progress'] = car['lap_progress']

    # 5. Large Reward for Completing a Lap (only given when completed in this step)
    if car.get('lap_completed_this_step', False):
        reward += 1000.0  # Give large reward for completing a lap

    # 6. Reverse Direction Penalty
    if car.get('is_going_wrong_way', False):
        reward -= 5.0

    # 7. 撞墙惩罚（Modified: Significantly increased penalty）
    if car.get('hit_wall_this_step', False):
        reward -= 50.0  # Changed from -20 to -50, strongly discourage wall collision

    # 8. 障碍物碰撞惩罚（Modified: Increased penalty）
    if car['collision_cooldown'] == 30 and not car.get('hit_wall_this_step', False):  # 刚发生碰撞但不是撞墙
        reward -= 20.0  # Changed from -8 to -20

    # 8.5 碰撞冷却期持续惩罚（New: Make AI remember wall collision pain）
    if car['collision_cooldown'] > 0:
        reward -= 0.5  # Extra -0.5 per step for 30 steps after collision

    # 9. 死亡惩罚（仅在血量刚降到0的瞬间触发一次）
    if car['health'] <= 0 and car['collision_cooldown'] == 30:
        reward -= 30.0

    # 10. Collect power-upReward（Single type, small bonus）
    if car['collected_power_up']:
        reward += 5  # Small bonus
        car['collected_power_up'] = False

    # 11. 新增：方向对齐Reward - 鼓励沿赛道切线方向行驶
    angle_to_center = math.atan2(dy, dx)
    ideal_direction = math.degrees(angle_to_center + math.pi / 2) % 360  # 顺时针切线方向
    current_direction = car['direction'] % 360

    # 计算方向差异（考虑360度循环）
    direction_diff = abs(ideal_direction - current_direction)
    if direction_diff > 180:
        direction_diff = 360 - direction_diff

    # 方向对齐Reward：方向越接近切线方向，Reward越高
    direction_alignment = 1.0 - (direction_diff / 180.0)  # 0到1之间
    reward += direction_alignment * 2.0  # 最高2分的方向对齐Reward

    return reward

def game_step(game_state, player_action=None, ai_actions=None):
    """Execute one game step"""
    rewards = {'player': 0, 'ai': [0] * len(game_state['ai_cars'])}
    dones = {'player': False, 'ai': [False] * len(game_state['ai_cars'])}

    game_state['time'] += 1

    track_params = (game_state['track_center_x'], game_state['track_center_y'],
                    game_state['track_width'], game_state['track_height'],
                    game_state.get('track_type', 'oval'))

    # 处理玩家Action
    if player_action is not None:
        game_state['player_car'] = update_car(
            game_state['player_car'], player_action, track_params,
            game_state['obstacles'], game_state['power_ups']
        )

    # 处理AIAction
    if ai_actions is not None:
        for i, (ai_car, action) in enumerate(zip(game_state['ai_cars'], ai_actions)):
            if action is not None:
                game_state['ai_cars'][i] = update_car(
                    ai_car, action, track_params,
                    game_state['obstacles'], game_state['power_ups']
                )

    # 检查碰撞
    all_cars = [game_state['player_car']] + game_state['ai_cars']

    # 赛车与障碍物碰撞
    for car in all_cars:
        for obstacle in game_state['obstacles']:
            if check_collision(car, obstacle):
                car = handle_collision(car)

    # 赛车之间碰撞
    for i in range(len(all_cars)):
        for j in range(i + 1, len(all_cars)):
            if check_collision(all_cars[i], all_cars[j]):
                all_cars[i] = handle_collision(all_cars[i])
                all_cars[j] = handle_collision(all_cars[j])

    # 检查道具收集
    for car in all_cars:
        for power_up in game_state['power_ups'][:]:
            if check_collision(car, power_up):
                car = collect_power_up(car, power_up)
                game_state['power_ups'].remove(power_up)

    # 计算Reward
    rewards['player'] = calculate_reward(game_state, game_state['player_car'])
    for i, ai_car in enumerate(game_state['ai_cars']):
        rewards['ai'][i] = calculate_reward(game_state, ai_car)

    # 检查是否有赛车率先完成5圈（获胜条件）
    if game_state['player_car']['laps_completed'] >= game_state['max_laps']:
        game_state['is_game_over'] = True
        game_state['winner'] = 'player'
        dones['player'] = True
        for i in range(len(game_state['ai_cars'])):
            dones['ai'][i] = True
    else:
        for i, ai_car in enumerate(game_state['ai_cars']):
            if ai_car['laps_completed'] >= game_state['max_laps']:
                game_state['is_game_over'] = True
                game_state['winner'] = f'AI{i}'
                dones['player'] = True
                for j in range(len(game_state['ai_cars'])):
                    dones['ai'][j] = True
                break

    return game_state, rewards, dones

def reset_game(game_state):
    """Reset game"""
    return create_game_state(
        width=game_state['width'],
        height=game_state['height'],
        num_ai_cars=game_state['num_ai_cars']
    )

### DQN

In [3]:
# @title 3.1 DQN 网络和缓冲区函数

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

def create_dqn_network(state_size=36, action_size=4, hidden_size=256):
    """Create DQN network"""
    import torch.nn as nn

    class DQN(nn.Module):
        def __init__(self, state_size, action_size, hidden_size):
            super(DQN, self).__init__()
            self.fc1 = nn.Linear(state_size, hidden_size)
            self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
            self.fc3 = nn.Linear(hidden_size // 2, hidden_size // 4)
            self.fc4 = nn.Linear(hidden_size // 4, action_size)

            # 初始化权重
            self._initialize_weights()

        def _initialize_weights(self):
            for module in self.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    nn.init.constant_(module.bias, 0.01)

        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = F.relu(self.fc3(x))
            return self.fc4(x)

    return DQN(state_size, action_size, hidden_size)

def create_replay_buffer(capacity=10000):
    """Create replay buffer"""
    return {
        'buffer': deque(maxlen=capacity),
        'capacity': capacity
    }

def push_to_buffer(buffer, state, action, reward, next_state, done):
    """Store experience to buffer"""
    buffer['buffer'].append((state, action, reward, next_state, done))

def sample_from_buffer(buffer, batch_size):
    """Sample from buffer"""
    if len(buffer['buffer']) < batch_size:
        return None

    batch = random.sample(list(buffer['buffer']), batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)

    return (
        torch.FloatTensor(np.array(states)),
        torch.LongTensor(np.array(actions)),
        torch.FloatTensor(np.array(rewards)).unsqueeze(1),
        torch.FloatTensor(np.array(next_states)),
        torch.FloatTensor(np.array(dones)).unsqueeze(1)
    )

def buffer_size(buffer):
    """Get buffer size"""
    return len(buffer['buffer'])

# @title 3.2 DQN 智能体函数

def create_dqn_agent(state_size=36, action_size=4):
    """Create DQN agent"""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 创建网络
    policy_net = create_dqn_network(state_size, action_size)
    target_net = create_dqn_network(state_size, action_size)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    # 移动到Device
    policy_net = policy_net.to(device)
    target_net = target_net.to(device)

    # 创建优化器
    optimizer = optim.Adam(policy_net.parameters(), lr=0.001)

    # 创建经验回放
    memory = create_replay_buffer(capacity=50000)

    return {
        'policy_net': policy_net,
        'target_net': target_net,
        'optimizer': optimizer,
        'memory': memory,
        'device': device,
        'state_size': state_size,
        'action_size': action_size,
        'gamma': 0.95,
        'epsilon': 1.0,
        'epsilon_min': 0.05,
        'epsilon_decay': 0.990,  # 方案C：加快探索衰减（从0.995改为0.990）
        'batch_size': 128,
        'target_update': 200,
        'steps_done': 0,
        'loss_history': [],
        'reward_history': []
    }

def select_action(agent, state, training=True, eval_epsilon=0.0):
    """选择Action"""
    agent['steps_done'] += 1

    # 探索（训练时使用agent['epsilon']，Evaluation时使用eval_epsilon）
    epsilon = agent['epsilon'] if training else eval_epsilon
    if random.random() < epsilon:
        return random.randint(0, agent['action_size'] - 1)

    # 利用
    with torch.no_grad():
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(agent['device'])
        q_values = agent['policy_net'](state_tensor)
        action = q_values.argmax().item()

    return action

def optimize_model(agent):
    """Optimize model"""
    batch = sample_from_buffer(agent['memory'], agent['batch_size'])
    if batch is None:
        return None

    states, actions, rewards, next_states, dones = batch

    # 移动到Device
    states = states.to(agent['device'])
    actions = actions.to(agent['device'])
    rewards = rewards.to(agent['device'])
    next_states = next_states.to(agent['device'])
    dones = dones.to(agent['device'])

    # 计算当前Q值
    current_q_values = agent['policy_net'](states).gather(1, actions.unsqueeze(1))

    # 计算目标Q值
    with torch.no_grad():
        next_q_values = agent['target_net'](next_states).max(1)[0].unsqueeze(1)
        target_q_values = rewards + (1 - dones) * agent['gamma'] * next_q_values

    # 计算损失
    loss = F.smooth_l1_loss(current_q_values, target_q_values)

    # 优化
    agent['optimizer'].zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(agent['policy_net'].parameters(), 1.0)
    agent['optimizer'].step()

    agent['loss_history'].append(loss.item())

    return loss.item()

def update_target_network(agent):
    """Update target network"""
    agent['target_net'].load_state_dict(agent['policy_net'].state_dict())

def update_epsilon(agent):
    """更新Epsilon"""
    agent['epsilon'] = max(agent['epsilon_min'], agent['epsilon'] * agent['epsilon_decay'])

def remember_experience(agent, state, action, reward, next_state, done):
    """Store experience"""
    push_to_buffer(agent['memory'], state, action, reward, next_state, done)

def save_agent(agent, path):
    """Save agent"""
    torch.save({
        'policy_net_state_dict': agent['policy_net'].state_dict(),
        'target_net_state_dict': agent['target_net'].state_dict(),
        'optimizer_state_dict': agent['optimizer'].state_dict(),
        'epsilon': agent['epsilon'],
        'steps_done': agent['steps_done'],
        'loss_history': agent['loss_history'],
        'reward_history': agent['reward_history']
    }, path)

def load_agent(agent, path):
    """Load agent"""
    checkpoint = torch.load(path, map_location=agent['device'])
    agent['policy_net'].load_state_dict(checkpoint['policy_net_state_dict'])
    agent['target_net'].load_state_dict(checkpoint['target_net_state_dict'])
    agent['optimizer'].load_state_dict(checkpoint['optimizer_state_dict'])
    agent['epsilon'] = checkpoint['epsilon']
    agent['steps_done'] = checkpoint['steps_done']
    agent['loss_history'] = checkpoint['loss_history']
    agent['reward_history'] = checkpoint['reward_history']

# @title 3.3 多智能体管理函数

def create_multi_agent_manager(num_agents=3, state_size=36, action_size=4):
    """Create multi-agent manager"""
    agents = [create_dqn_agent(state_size, action_size) for _ in range(num_agents)]

    return {
        'agents': agents,
        'num_agents': num_agents,
        'episode_rewards': [[] for _ in range(num_agents)],
        'episode_lengths': [[] for _ in range(num_agents)]
    }

def select_actions(manager, states, training=True, eval_epsilon=0.0):
    """为所有智能体选择Action"""
    actions = []
    for i, (agent, state) in enumerate(zip(manager['agents'], states)):
        action = select_action(agent, state, training, eval_epsilon)
        actions.append(action)
    return actions

def remember_experiences(manager, states, actions, rewards, next_states, dones):
    """为所有智能体Store experience"""
    for i, agent in enumerate(manager['agents']):
        if i < len(states):
            remember_experience(agent, states[i], actions[i], rewards[i], next_states[i], dones[i])

def optimize_all_agents(manager):
    """Optimize all agents（不在此处更新epsilon，改为每episode结束后更新）"""
    losses = []
    for agent in manager['agents']:
        loss = optimize_model(agent)
        if loss is not None:
            losses.append(loss)

    # Update target network
    if manager['agents'][0]['steps_done'] % manager['agents'][0]['target_update'] == 0:
        for agent in manager['agents']:
            update_target_network(agent)

    return np.mean(losses) if losses else None

def update_statistics(manager, rewards, steps):
    """Update training statistics"""
    for i in range(manager['num_agents']):
        if i < len(rewards):
            manager['episode_rewards'][i].append(rewards[i])
            manager['episode_lengths'][i].append(steps)

def get_training_stats(manager):
    """Get training statistics"""
    stats = {
        'avg_rewards': [np.mean(rewards[-100:]) if rewards else 0
                       for rewards in manager['episode_rewards']],
        'avg_lengths': [np.mean(lengths[-100:]) if lengths else 0
                       for lengths in manager['episode_lengths']],
        'epsilons': [agent['epsilon'] for agent in manager['agents']],
        'memory_sizes': [buffer_size(agent['memory']) for agent in manager['agents']]
    }
    return stats

def save_all_agents(manager, base_path):
    """Save all agents"""
    for i, agent in enumerate(manager['agents']):
        save_agent(agent, f"{base_path}_agent_{i}.pth")

def load_all_agents(manager, base_path):
    """Load all agents"""
    for i, agent in enumerate(manager['agents']):
        load_agent(agent, f"{base_path}_agent_{i}.pth")

def find_best_model(num_agents):
    """Find best local model，返回base_path或None"""
    import glob, re
    files = glob.glob('best_model_ep*_agent_0.pth')
    if not files:
        return None
    # 提取Episode数，选最大的
    best_ep, best_path = 0, None
    for f in files:
        m = re.search(r'best_model_ep(\d+)_agent_0\.pth', f)
        if m:
            ep = int(m.group(1))
            # 检查所有agent文件都存在
            base = f'best_model_ep{ep}'
            if all(os.path.exists(f'{base}_agent_{i}.pth') for i in range(num_agents)):
                if ep > best_ep:
                    best_ep, best_path = ep, base
    return best_path

### Training Functions\n

In [4]:
def collect_random_experiences(game_state, manager, num_steps=1000):
    """Collecting random experiences填充缓冲区"""
    state = copy.deepcopy(game_state)

    for step in range(num_steps):
        # 获取状态
        states = [get_state_vector(state, car) for car in state['ai_cars']]

        # 随机Action（修改：从0-3，因为取消了加速Action）
        ai_actions = [random.randint(0, 3) for _ in range(state['num_ai_cars'])]

        # 执行游戏步长
        state, rewards, dones = game_step(state, 0, ai_actions)

        # 获取下一个状态
        next_states = [get_state_vector(state, car) for car in state['ai_cars']]

        # Store experience
        remember_experiences(manager, states, ai_actions, rewards['ai'], next_states, dones['ai'])

        # 如果游戏结束，重置
        if all(dones['ai']):
            state = reset_game(state)

    return state, manager

def train_episode(game_state, manager, episode_num, max_steps=None):
    """Train oneRound（max_steps=None表示No-op限制，靠完成5圈结束）"""
    state = reset_game(game_state)

    # 获取初始状态
    states = [get_state_vector(state, car) for car in state['ai_cars']]

    episode_rewards = [0] * manager['num_agents']
    step = 0
    done = False

    while not done and (max_steps is None or step < max_steps):
        step += 1

        # 选择Action
        ai_actions = select_actions(manager, states, training=True)

        # 执行游戏步长
        state, rewards, dones = game_step(state, 0, ai_actions)

        # 训练时自动复活死亡的AI（重置到起跑线，但保留已完成圈数）
        for i, car in enumerate(state['ai_cars']):
            if car['health'] <= 0:
                car['health'] = 100
                car['speed'] = 3.0
                # 重置到起跑线位置
                if 'start_positions' in state and i < len(state['start_positions']['ai']):
                    car['x'], car['y'] = state['start_positions']['ai'][i]
                    car['direction'] = state.get('initial_direction', 270)
                # 重置角度进度（但不重置laps_completed）
                car['prev_angle'] = None
                car['cumulative_angle'] = 0.0
                car['lap_progress'] = 0

        # 获取下一个状态
        next_states = [get_state_vector(state, car) for car in state['ai_cars']]

        # 直接使用 game_step 中 calculate_reward 计算的Reward
        modified_rewards = rewards['ai']

        # Store experience
        remember_experiences(manager, states, ai_actions, modified_rewards, next_states, dones['ai'])

        # Optimize model
        if all(buffer_size(agent['memory']) >= agent['batch_size'] for agent in manager['agents']):
            optimize_all_agents(manager)

        # 更新Reward
        for i in range(manager['num_agents']):
            episode_rewards[i] += modified_rewards[i]

        # 更新状态
        states = next_states

        # 如果所有AI都停了，提前结束
        if all(dones['ai']) or state['is_game_over']:
            done = True

    # 每个episode结束后衰减Epsilon（而非每步衰减）
    for agent in manager['agents']:
        update_epsilon(agent)

    # 更新统计
    update_statistics(manager, episode_rewards, step)
    stats = get_training_stats(manager)

    return state, manager, np.mean(episode_rewards), step, stats

def evaluate_model(game_state, manager, num_episodes=5, max_steps=None, eval_epsilon=0.05):
    """Evaluation模型（max_steps=None表示No-op限制，eval_epsilon为Evaluation时的Epsilon）"""
    total_rewards = []

    for ep in range(num_episodes):
        state = reset_game(game_state)

        states = [get_state_vector(state, car) for car in state['ai_cars']]
        episode_reward = 0
        step = 0
        done = False

        while not done and (max_steps is None or step < max_steps):
            step += 1

            # 选择Action（方案C：使用小量Epsiloneval_epsilon=0.05）
            ai_actions = select_actions(manager, states, training=False, eval_epsilon=eval_epsilon)

            # 执行游戏步长
            state, rewards, dones = game_step(state, 0, ai_actions)

            # 获取下一个状态
            next_states = [get_state_vector(state, car) for car in state['ai_cars']]

            # 更新Reward
            episode_reward += np.mean(rewards['ai'])

            # 更新状态
            states = next_states

            if all(dones['ai']) or state['is_game_over']:
                done = True

        total_rewards.append(episode_reward)

    return {
        'mean_reward': np.mean(total_rewards),
        'std_reward': np.std(total_rewards)
    }


In [5]:
def train(config=None):
    """Main training function - 修复版"""
    default_config = {
        'num_episodes': 600,  # 设置为600Episode
        'max_steps_per_episode': 2000,  #
        'num_ai_cars': 3,
        'game_width': 800,
        'game_height': 600,
        'eval_interval': 20,
        'num_eval_episodes': 3,
        'save_interval': 50
    }
    if config is not None:
        default_config.update(config)
    config = default_config

    print("=" * 60)
    print("🏎️ AI Racing Game - Training started")
    print("=" * 60)
    print(f"Config: {config}")
    print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    print("=" * 60)

    # Create game state
    game_state = create_game_state(
        width=config['game_width'],
        height=config['game_height'],
        num_ai_cars=config['num_ai_cars']
    )

    # Create multi-agent manager
    manager = create_multi_agent_manager(
        num_agents=config['num_ai_cars'],
        state_size=36,  # 7+10+15+4=36维
        action_size=4  # No-op操作、Brake、Left、Right
    )

    # Collecting random experiences
    print("Collecting random experiences...")
    game_state, manager = collect_random_experiences(game_state, manager, num_steps=1000)
    print(f"✅ Collection complete")

    # 训练记录
    history = {
        'episode_rewards': [],
        'episode_lengths': [],
        'eval_rewards': [],
        'epsilons': []
    }

    best_reward = -np.inf

    # 训练循环
    for episode in range(1, config['num_episodes'] + 1):
        # Train oneEpisode
        game_state, manager, reward, steps, stats = train_episode(
            game_state, manager, episode,
            max_steps=config['max_steps_per_episode']
        )

        # 记录历史
        history['episode_rewards'].append(reward)
        history['episode_lengths'].append(steps)
        history['epsilons'].append(np.mean(stats['epsilons']))

        # 打印进度
        if episode % 10 == 0:
            avg_reward = np.mean(history['episode_rewards'][-10:]) if len(history['episode_rewards']) >= 10 else reward
            print(f"Episode {episode:3d} | Reward: {reward:6.2f} | 平均Reward: {avg_reward:6.2f} | Steps: {steps:3d} | Epsilon: {stats['epsilons'][0]:.3f}")

        # Evaluation
        if episode % config['eval_interval'] == 0:
            eval_result = evaluate_model(
                game_state, manager,
                num_episodes=config['num_eval_episodes'],
                max_steps=config['max_steps_per_episode']
            )
            history['eval_rewards'].append(eval_result['mean_reward'])
            print(f"📊 Evaluation | Avg Reward: {eval_result['mean_reward']:.2f} ± {eval_result['std_reward']:.2f}")

            if eval_result['mean_reward'] > best_reward:
                best_reward = eval_result['mean_reward']
                # 保存最佳模型
                save_all_agents(manager, f"best_model_ep{episode}")
                print(f"🏆 New best model! Reward: {best_reward:.2f}")

    print("\n" + "=" * 60)
    print("✅ Training completed!")
    print("=" * 60)

    return manager, history

### Visualization\n

In [6]:

"""## 📊 5. 可视化模块"""

# @title 5.1 Train 历史可视化

import matplotlib.pyplot as plt

def plot_training_history(history):
    """绘制Train 历史"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # 1. Reward曲线
    axes[0, 0].plot(history['episode_rewards'], alpha=0.6, label='RoundReward')
    if len(history['episode_rewards']) > 10:
        moving_avg = np.convolve(history['episode_rewards'],
                                np.ones(10)/10, mode='valid')
        axes[0, 0].plot(range(9, len(history['episode_rewards'])),
                      moving_avg, 'r-', linewidth=2, label='10Round移动平均')
    axes[0, 0].set_xlabel('Round')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].set_title('Train Reward')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Round长度
    axes[0, 1].plot(history['episode_lengths'], alpha=0.6)
    axes[0, 1].set_xlabel('Round')
    axes[0, 1].set_ylabel('Step')
    axes[0, 1].set_title('Round长度')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. EvaluationReward - 修复这里：使用history中的长度信息
    if history.get('eval_rewards') and len(history['eval_rewards']) > 0:
        # 计算Evaluation发生的Round
        eval_interval = len(history['episode_rewards']) // len(history['eval_rewards'])
        if eval_interval > 0:
            eval_episodes = range(eval_interval,
                                 len(history['episode_rewards']) + 1,
                                 eval_interval)
            # 确保长度匹配
            eval_episodes = list(eval_episodes)[:len(history['eval_rewards'])]
            axes[1, 0].plot(eval_episodes, history['eval_rewards'], 'g-o')
        else:
            axes[1, 0].plot(history['eval_rewards'], 'g-o')

        axes[1, 0].set_xlabel('Round')
        axes[1, 0].set_ylabel('EvaluationReward')
        axes[1, 0].set_title('Evaluation性能')
        axes[1, 0].grid(True, alpha=0.3)

    # 4. Exploration rate 衰减
    if history.get('epsilons'):
        axes[1, 1].plot(history['epsilons'])
        axes[1, 1].set_xlabel('Round')
        axes[1, 1].set_ylabel('Exploration rate ')
        axes[1, 1].set_title('Exploration rate 衰减')
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return fig

def plot_comparison(history1, history2, label1="Train 1", label2="Train 2"):
    """比较两次Train 的结果"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Reward比较
    axes[0].plot(history1['episode_rewards'], alpha=0.6, label=label1)
    axes[0].plot(history2['episode_rewards'], alpha=0.6, label=label2)
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Reward')
    axes[0].set_title('Train Reward比较')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # 移动平均比较
    if len(history1['episode_rewards']) > 50:
        ma1 = np.convolve(history1['episode_rewards'], np.ones(50)/50, mode='valid')
        ma2 = np.convolve(history2['episode_rewards'], np.ones(50)/50, mode='valid')
        axes[1].plot(range(49, len(history1['episode_rewards'])), ma1, linewidth=2, label=label1)
        axes[1].plot(range(49, len(history2['episode_rewards'])), ma2, linewidth=2, label=label2)
        axes[1].set_xlabel('Round')
        axes[1].set_ylabel('50Round移动平均')
        axes[1].set_title('移动平均比较')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return fig

In [7]:
def debug_ai_behavior(game_state, manager, num_steps=50):
    """Debug AI behavior"""
    print("\nDebug AI behavior")
    print("=" * 60)

    state = copy.deepcopy(game_state)
    step_data = []

    for step in range(num_steps):
        # 获取状态
        states = [get_state_vector(state, car) for car in state['ai_cars']]

        # 选择Action
        actions = select_actions(manager, states, training=False)

        # 记录每个AI的状态
        for i, (car, action) in enumerate(zip(state['ai_cars'], actions)):
            step_data.append({
                'step': step,
                'ai_id': i,
                'action': action,
                'speed': car['speed'],
                'health': car['health'],
                'x': car['x'],
                'y': car['y']
            })

        # 执行步长
        state, rewards, dones = game_step(state, 0, actions)

        if step % 10 == 0:
            print(f"Step {step}: Action={actions}, Speed={[c['speed'] for c in state['ai_cars']]}, Health={[c['health'] for c in state['ai_cars']]}")

        if all(dones['ai']):
            print(f"Stop at Step {step} (AI cars finished)")
            break

    return step_data

### Visualization

In [8]:
import pygame

# 颜色常量
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GRAY = (100, 100, 100)
DARK_GRAY = (50, 50, 50)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
YELLOW = (255, 255, 0)
BLUE = (0, 150, 255)
ORANGE = (255, 100, 0)

def draw_track(screen, game_state):
    """Draw track - 根据track_type绘制不同形状"""
    cx = game_state['track_center_x']
    cy = game_state['track_center_y']
    tw = game_state['track_width']
    th = game_state['track_height']
    track_type = game_state.get('track_type', 'oval')
    inner_ratio = 0.5
    
    # 根据赛道类型生成外圈和内圈的点
    outer_points = []
    inner_points = []
    
    for angle in np.linspace(0, 2 * math.pi, 100):
        if track_type == 'oval':
            r_factor = 1.0
        elif track_type == 'peanut':
            r_factor = 1 + 0.3 * math.sin(2 * angle)
        elif track_type == 'kidney':
            r_factor = 1 + 0.2 * math.sin(angle) + 0.15 * math.cos(3 * angle)
        else:
            r_factor = 1.0
        
        # 外圈点
        x_outer = cx + (tw / 2) * r_factor * math.cos(angle)
        y_outer = cy + (th / 2) * r_factor * math.sin(angle)
        outer_points.append((int(x_outer), int(y_outer)))
        
        # 内圈点
        x_inner = cx + (tw * inner_ratio / 2) * r_factor * math.cos(angle)
        y_inner = cy + (th * inner_ratio / 2) * r_factor * math.sin(angle)
        inner_points.append((int(x_inner), int(y_inner)))
    
    # 绘制赛道（外圈填充灰色）
    pygame.draw.polygon(screen, GRAY, outer_points, 0)
    # 内圈挖空（填充黑色）
    pygame.draw.polygon(screen, BLACK, inner_points, 0)
    # 绘制边线
    pygame.draw.polygon(screen, WHITE, outer_points, 2)
    pygame.draw.polygon(screen, WHITE, inner_points, 2)

    # 起点/终点线（位于45°-60°扇形区域，与赛道前进方向呈30°夹角）
    # 使用52.5°作为终点线的中心角度（45°和60°的中点）
    finish_angle = math.radians(52.5)  # 从右侧顺时针52.5°
    line_angle = finish_angle + math.radians(30)  # 终点线与径向呈30°夹角

    # 计算该角度的r_factor
    if track_type == 'oval':
        r_factor_finish = 1.0
    elif track_type == 'peanut':
        r_factor_finish = 1 + 0.3 * math.sin(2 * finish_angle)
    elif track_type == 'kidney':
        r_factor_finish = 1 + 0.2 * math.sin(finish_angle) + 0.15 * math.cos(3 * finish_angle)
    else:
        r_factor_finish = 1.0
    
    # 赛道外圈和内圈半径（考虑r_factor）
    outer_radius_x = (tw / 2) * r_factor_finish
    outer_radius_y = (th / 2) * r_factor_finish
    inner_radius_x = (tw * inner_ratio / 2) * r_factor_finish
    inner_radius_y = (th * inner_ratio / 2) * r_factor_finish

    # 计算终点线在finish_angle方向上的外圈和内圈点
    outer_x = cx + outer_radius_x * math.cos(finish_angle)
    outer_y = cy + outer_radius_y * math.sin(finish_angle)
    inner_x = cx + inner_radius_x * math.cos(finish_angle)
    inner_y = cy + inner_radius_y * math.sin(finish_angle)

    # 终点线长度（从外圈到内圈的实际距离）
    line_length = math.sqrt((outer_x - inner_x)**2 + (outer_y - inner_y)**2)

    # 终点线中心点（外圈和内圈的中点）
    center_x = (outer_x + inner_x) / 2
    center_y = (outer_y + inner_y) / 2

    # 终点线的两个端点
    half_len = line_length / 2
    dx = half_len * math.cos(line_angle)
    dy = half_len * math.sin(line_angle)

    p1_x = int(center_x - dx)
    p1_y = int(center_y - dy)
    p2_x = int(center_x + dx)
    p2_y = int(center_y + dy)

    # 绘制粗黄线作为起点/终点
    for offset in range(-3, 4):  # 绘制7条线形成粗线效果
        offset_dx = offset * math.cos(line_angle + math.pi/2)
        offset_dy = offset * math.sin(line_angle + math.pi/2)
        pygame.draw.line(screen, YELLOW,
                        (int(p1_x + offset_dx), int(p1_y + offset_dy)),
                        (int(p2_x + offset_dx), int(p2_y + offset_dy)), 3)

    # 绘制"START/FINISH"文字
    font_large = pygame.font.SysFont(None, 28, bold=True)
    start_finish_text = font_large.render('START/FINISH', True, YELLOW)
    screen.blit(start_finish_text, (int(center_x + 10), int(center_y - 20)))

def draw_car(screen, car, font):
    """Draw car"""
    x, y = int(car['x']), int(car['y'])
    color = car['color']
    # 赛车主体
    pygame.draw.rect(screen, color, (x - car['width']//2, y - car['height']//2, car['width'], car['height']))
    # 方向指示
    dx = int(15 * math.cos(math.radians(car['direction'])))
    dy = int(15 * math.sin(math.radians(car['direction'])))
    pygame.draw.line(screen, WHITE, (x, y), (x + dx, y + dy), 2)
    # 护盾效果
    if car['has_shield']:
        pygame.draw.circle(screen, BLUE, (x, y), 20, 2)
    # 血条
    bar_w = car['width']
    bar_h = 4
    bar_x = x - bar_w // 2
    bar_y = y - car['height'] // 2 - 8
    pygame.draw.rect(screen, RED, (bar_x, bar_y, bar_w, bar_h))
    pygame.draw.rect(screen, GREEN, (bar_x, bar_y, int(bar_w * car['health'] / 100), bar_h))

def draw_obstacles(screen, obstacles):
    """Draw obstacles"""
    for obs in obstacles:
        pygame.draw.rect(screen, obs['color'],
                         (int(obs['x'] - obs['size']//2), int(obs['y'] - obs['size']//2),
                          obs['size'], obs['size']))

def draw_power_ups(screen, power_ups):
    """Draw power-ups"""
    for pu in power_ups:
        pygame.draw.circle(screen, pu['color'], (int(pu['x']), int(pu['y'])), pu['size'])

def draw_hud(screen, font, game_state, rewards_info=None):
    """Draw HUD"""
    y_offset = 10
    # 时间
    text = font.render(f'Time: {game_state["time"]}', True, WHITE)
    screen.blit(text, (10, y_offset))
    y_offset += 25
    # 玩家信息
    pc = game_state['player_car']
    text = font.render(f'Player - HP:{pc["health"]} Speed:{pc["speed"]:.1f} Laps:{pc["laps_completed"]}', True, GREEN)
    screen.blit(text, (10, y_offset))
    y_offset += 25
    # AI信息
    for i, car in enumerate(game_state['ai_cars']):
        text = font.render(f'AI{i} - HP:{car["health"]} Speed:{car["speed"]:.1f} Laps:{car["laps_completed"]}', True, RED)
        screen.blit(text, (10, y_offset))
        y_offset += 20
    # Reward信息
    if rewards_info:
        text = font.render(f'Reward: {rewards_info:.2f}', True, YELLOW)
        screen.blit(text, (10, y_offset))

def run_game_with_ui(manager=None, max_steps=1500, num_ai_cars=3):
    """Run game with UI（按ESC或关闭窗口退出）"""
    pygame.init()
    width, height = 1000, 750
    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption('AI Racing Game - Q-Learning (ESC to quit)')
    clock = pygame.time.Clock()
    font = pygame.font.SysFont(None, 22)
    big_font = pygame.font.SysFont(None, 48)

    # 如果没有训练好的manager，创建一个新的
    if manager is None:
        manager = create_multi_agent_manager(num_agents=num_ai_cars, state_size=36, action_size=4)

    # 多Map系统
    maps = ['oval', 'peanut']  # 2张Map
    current_map = 0
    car_scores = [0] * num_ai_cars  # 每辆AI车的Total Score
    player_score = 0  # 玩家的Total Score

    # 游戏状态
    game_started = False
    countdown = 3
    countdown_timer = 0
    running = True
    action_names = ['No-op', 'Brake', 'Left', 'Right']

    # 创建初始游戏状态
    game_state = create_game_state(width=width, height=height, num_ai_cars=num_ai_cars, track_type=maps[current_map])
    step = 0
    total_reward = 0

    # 等待开始按钮
    while running and not game_started:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                mouse_pos = pygame.mouse.get_pos()
                # START按钮区域 (中心位置)
                button_rect = pygame.Rect(width//2 - 100, height//2 - 30, 200, 60)
                if button_rect.collidepoint(mouse_pos):
                    game_started = True
                    countdown_timer = pygame.time.get_ticks()

        # 绘制等待界面
        screen.fill(BLACK)
        draw_track(screen, game_state)

        # 绘制START按钮
        button_rect = pygame.Rect(width//2 - 100, height//2 - 30, 200, 60)
        pygame.draw.rect(screen, GREEN, button_rect)
        pygame.draw.rect(screen, WHITE, button_rect, 3)
        start_text = big_font.render('START', True, BLACK)
        text_rect = start_text.get_rect(center=button_rect.center)
        screen.blit(start_text, text_rect)

        # 提示文字
        hint_text = font.render('Click START to begin', True, WHITE)
        screen.blit(hint_text, (width//2 - 100, height//2 + 50))

        pygame.display.flip()
        clock.tick(30)

    # 倒计时阶段
    while running and countdown > 0:
        current_time = pygame.time.get_ticks()
        if current_time - countdown_timer >= 1000:
            countdown -= 1
            countdown_timer = current_time

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False

        # 绘制倒计时
        screen.fill(BLACK)
        draw_track(screen, game_state)
        draw_obstacles(screen, game_state['obstacles'])
        draw_power_ups(screen, game_state['power_ups'])
        draw_car(screen, game_state['player_car'], font)
        for car in game_state['ai_cars']:
            draw_car(screen, car, font)

        # 显示倒计时数字
        countdown_text = big_font.render(str(countdown), True, YELLOW)
        text_rect = countdown_text.get_rect(center=(width//2, height//2))
        screen.blit(countdown_text, text_rect)

        pygame.display.flip()
        clock.tick(30)

    # 主游戏循环
    while running and step < max_steps:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False

        # 玩家键盘控制 - 支持WASD和方向键（取消加速）
        keys = pygame.key.get_pressed()
        player_action = 0
        if keys[pygame.K_DOWN] or keys[pygame.K_s]: player_action = 1  # Brake
        elif keys[pygame.K_LEFT] or keys[pygame.K_a]: player_action = 2  # Left
        elif keys[pygame.K_RIGHT] or keys[pygame.K_d]: player_action = 3  # Right

        # AIAction
        ai_states = [get_state_vector(game_state, car) for car in game_state['ai_cars']]
        ai_actions = select_actions(manager, ai_states, training=False)

        # 执行游戏步长
        game_state, rewards, dones = game_step(game_state, player_action, ai_actions)
        total_reward += np.mean(rewards['ai'])

        # 累积每辆车的分数（包括玩家）
        player_score += rewards['player']
        for i, reward in enumerate(rewards['ai']):
            car_scores[i] += reward

        step += 1

        # 检查是否有任意赛车（包括玩家）完成3圈
        all_cars = [game_state['player_car']] + game_state['ai_cars']
        any_car_finished = any(car['laps_completed'] >= 3 for car in all_cars)
        if any_car_finished:
            current_map += 1
            if current_map >= len(maps):
                # 所有Map完成，显示Scoreboard
                break
            else:
                # 切换到下一张Map
                print(f"\nMap{current_map}完成！切换到Map{current_map+1}...")
                game_state = create_game_state(width=width, height=height, num_ai_cars=num_ai_cars, track_type=maps[current_map])
                step = 0

        # 如果有AI死亡，重置到起跑线位置（保留已完成圈数）
        for i, car in enumerate(game_state['ai_cars']):
            if car['health'] <= 0:
                car['health'] = 100
                car['speed'] = 3.0
                # 重置到起跑线位置
                if 'start_positions' in game_state and i < len(game_state['start_positions']['ai']):
                    car['x'], car['y'] = game_state['start_positions']['ai'][i]
                    car['direction'] = game_state.get('initial_direction', 270)
                # 重置角度进度（但不重置laps_completed）
                car['prev_angle'] = None
                car['cumulative_angle'] = 0.0
                car['lap_progress'] = 0
        if game_state['player_car']['health'] <= 0:
            game_state['player_car']['health'] = 100
            game_state['player_car']['speed'] = 3.0
            # 重置到起跑线位置（保留已完成圈数）
            if 'start_positions' in game_state:
                game_state['player_car']['x'], game_state['player_car']['y'] = game_state['start_positions']['player']
                game_state['player_car']['direction'] = game_state.get('initial_direction', 270)
            # 重置角度进度（但不重置laps_completed）
            game_state['player_car']['prev_angle'] = None
            game_state['player_car']['cumulative_angle'] = 0.0
            game_state['player_car']['lap_progress'] = 0

        # 绘制
        screen.fill(BLACK)
        draw_track(screen, game_state)
        draw_obstacles(screen, game_state['obstacles'])
        draw_power_ups(screen, game_state['power_ups'])
        draw_car(screen, game_state['player_car'], font)
        for car in game_state['ai_cars']:
            draw_car(screen, car, font)
        draw_hud(screen, font, game_state, rewards_info=total_reward/max(step,1))

        # 显示Steps、Map信息和AIAction
        step_text = font.render(f'Step: {step}/{max_steps}', True, WHITE)
        screen.blit(step_text, (width - 150, 10))
        map_text = font.render(f'Map: {current_map+1}/{len(maps)} ({maps[current_map]})', True, YELLOW)
        screen.blit(map_text, (width - 200, 35))
        action_text = font.render(f'AI Actions: {[action_names[a] for a in ai_actions]}', True, WHITE)
        screen.blit(action_text, (10, height - 30))

        # 显示Player Control说明（右下角）
        control_font = pygame.font.SysFont(None, 20)
        controls = [
            'Player Control (Green Car):',
            '↓/S Brake',
            '←/A Left  →/D Right',
            'ESC Exit Game'
        ]
        y_start = height - 120
        for i, text in enumerate(controls):
            color = GREEN if i == 0 else WHITE
            control_text = control_font.render(text, True, color)
            screen.blit(control_text, (width - 180, y_start + i * 22))

        pygame.display.flip()
        clock.tick(game_state['fps'])

    # 显示Scoreboard
    if current_map >= len(maps):
        # 创建排行榜（包括玩家）
        rankings = [('Player', player_score)] + [(i, score) for i, score in enumerate(car_scores)]
        rankings.sort(key=lambda x: x[1], reverse=True)

        # 显示排行榜界面
        showing_rankings = True
        while showing_rankings and running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                    showing_rankings = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        showing_rankings = False

            screen.fill(BLACK)

            # 标题
            title_text = big_font.render('Race Finished - Scoreboard', True, YELLOW)
            screen.blit(title_text, (width//2 - 200, 50))

            # 排行榜
            y_offset = 150
            for rank, (car_id, score) in enumerate(rankings, 1):
                if rank == 1:
                    color = YELLOW
                    medal = '🥇'
                elif rank == 2:
                    color = WHITE
                    medal = '🥈'
                elif rank == 3:
                    color = (205, 127, 50)  # 铜色
                    medal = '🥉'
                else:
                    color = WHITE
                    medal = ''

                car_name = 'Player (You)' if car_id == 'Player' else f'AI Car{car_id}'
                rank_text = font.render(f'{medal} Rank{rank}: {car_name} - Total Score: {score:.2f}', True, color)
                screen.blit(rank_text, (width//2 - 150, y_offset))
                y_offset += 40

            # 提示
            hint_text = font.render('Press ESC to exit', True, WHITE)
            screen.blit(hint_text, (width//2 - 60, height - 50))

            pygame.display.flip()
            clock.tick(30)

    pygame.quit()
    print(f'Game over - Steps: {step}, Avg AIReward: {total_reward/max(step,1):.2f}')
    if current_map >= len(maps):
        print('\n=== Scoreboard ===')
        rankings = [('Player', player_score)] + [(i, score) for i, score in enumerate(car_scores)]
        rankings.sort(key=lambda x: x[1], reverse=True)
        for rank, (car_id, score) in enumerate(rankings, 1):
            car_name = 'Player (You)' if car_id == 'Player' else f'AI Car{car_id}'
            print(f'Rank{rank}: {car_name} - Total Score: {score:.2f}')

pygame 2.6.1 (SDL 2.28.4, Python 3.12.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


### Test Functions\n

In [9]:

"""## 🚀 6. 测试模块"""

# @title 6.1 测试函数

def test_imports():
    """测试导入"""
    print("测试导入...")
    try:
        import torch
        import numpy as np
        import matplotlib.pyplot as plt
        print(f"✅ PyTorch版本: {torch.__version__}")
        print(f"✅ NumPy版本: {np.__version__}")
        print("✅ 所有库导入成功")
        return True
    except Exception as e:
        print(f"❌ 导入失败: {e}")
        return False

def test_game_environment():
    """测试游戏环境"""
    try:
        game_state = create_game_state()
        print(f"✅ Game state created")
        print(f"   - Number of AI cars: {len(game_state['ai_cars'])}")
        print(f"   - Number of obstacles: {len(game_state['obstacles'])}")
        print(f"   - Number of power-ups: {len(game_state['power_ups'])}")

        # 测试状态向量
        state = get_state_vector(game_state, game_state['player_car'])
        print(f"   - 状态向量维度: {len(state)}")

        # 测试游戏步长
        game_state, rewards, dones = game_step(game_state, 0, [0, 0, 0])
        print(f"✅ 游戏步长执行成功")

        return True
    except Exception as e:
        print(f"❌ 游戏环境测试失败: {e}")
        return False

def test_dqn_agent():
    """测试DQN智能体"""
    try:
        agent = create_dqn_agent()
        print(f"✅ DQN agent created")

        # 测试Action选择
        test_state = np.random.randn(41).astype(np.float32)
        action = select_action(agent, test_state)
        print(f"   - Action选择: {action}")

        # 测试经验存储
        next_state = np.random.randn(41).astype(np.float32)
        remember_experience(agent, test_state, action, 1.0, next_state, False)
        print(f"   - 经验存储成功")

        # 测试优化
        for _ in range(agent['batch_size']):
            remember_experience(agent,
                              np.random.randn(41).astype(np.float32),
                              random.randint(0, 4),
                              random.random(),
                              np.random.randn(41).astype(np.float32),
                              random.random() > 0.5)

        loss = optimize_model(agent)
        if loss is not None:
            print(f"   - 优化损失: {loss:.4f}")

        print("✅ DQN智能体测试通过")
        return True
    except Exception as e:
        print(f"❌ DQN智能体测试失败: {e}")
        return False

def test_multi_agent():
    """测试多智能体"""
    try:
        manager = create_multi_agent_manager(num_agents=3)
        print(f"✅ 多智能体管理器创建成功")

        # 测试状态
        states = [np.random.randn(41).astype(np.float32) for _ in range(3)]
        actions = select_actions(manager, states)
        print(f"   - Action选择: {actions}")

        # 测试经验存储
        next_states = [np.random.randn(41).astype(np.float32) for _ in range(3)]
        rewards = [1.0, 1.0, 1.0]
        dones = [False, False, False]
        remember_experiences(manager, states, actions, rewards, next_states, dones)
        print(f"   - 经验存储成功")

        print("✅ 多智能体测试通过")
        return True
    except Exception as e:
        print(f"❌ 多智能体测试失败: {e}")
        return False

def run_all_tests():
    """Running all tests"""
    print("=" * 60)
    print("🏁 Running all tests")
    print("=" * 60)

    tests = [
        ("导入测试", test_imports),
        ("游戏环境测试", test_game_environment),
        ("DQN智能体测试", test_dqn_agent),
        ("多智能体测试", test_multi_agent)
    ]

    results = []
    for name, test_func in tests:
        print(f"\n{'='*40}")
        print(f"测试: {name}")
        print('='*40)
        try:
            result = test_func()
            results.append(result)
        except Exception as e:
            print(f"❌ {name} 执行出错: {e}")
            results.append(False)

    print("\n" + "=" * 60)
    print("📊 测试结果汇总")
    print("=" * 60)
    for i, (name, _) in enumerate(tests):
        status = "✅ 通过" if results[i] else "❌ 失败"
        print(f"{name}: {status}")

    passed = sum(results)
    total = len(tests)
    print(f"\n通过率: {passed}/{total} ({passed/total*100:.1f}%)")

    return all(results)

### Main Function\n

In [10]:

def main(mode='demo'):
    """Main function"""
    print("=" * 60)
    print("🏎️ AI Racing Game - Multi-Agent Reinforcement Learning")
    print("=" * 60)

    if mode == 'full_train':
        # 完整训练模式（No-opUI）
        print("\nRunning full training mode...")
        config = {
            'num_episodes': 600,  # 设置为600Episode
            'max_steps_per_episode': 2000,  # 设置上限防止早期AI卡死
            'num_ai_cars': 3,
            'game_width': 800,
            'game_height': 600,
            'eval_interval': 20,
            'num_eval_episodes': 10,
            'save_interval': 50
        }
        manager, history = train(config)
        plot_training_history(history)

    elif mode == 'play':
        # Pygame UI模式 - 加载本地模型
        num_ai = 3
        best_path = find_best_model(num_ai)
        if best_path:
            print(f"\n📂 Found local model: {best_path}")
            manager = create_multi_agent_manager(num_agents=num_ai)
            load_all_agents(manager, best_path)
            print("✅ Model loaded successfully")
            print("\n🎮 Starting Pygame UI...")
            run_game_with_ui(manager=manager, max_steps=999999, num_ai_cars=num_ai)
        else:
            print("\n⚠️ Can not Found local model，Please run full_train mode first")

    else:  # demo模式
        print("\n🎮 Running demo mode...")
        game_state = create_game_state(num_ai_cars=2)
        print(f"✅ Game state created")
        print(f"   - Number of AI cars: {len(game_state['ai_cars'])}")
        print(f"   - Number of obstacles: {len(game_state['obstacles'])}")
        print(f"   - Number of power-ups: {len(game_state['power_ups'])}")

        player_state = get_state_vector(game_state, game_state['player_car'])
        print(f"   - Player state dimensions: {len(player_state)}")

        agent = create_dqn_agent()
        print(f"   - DQN agent created")

        print("\nExecuting demo steps...")
        for i in range(5):
            action = select_action(agent, player_state, training=True)
            game_state, rewards, dones = game_step(game_state, action, [0, 0])
            player_state = get_state_vector(game_state, game_state['player_car'])
            print(f"   步长 {i+1}: Action={action}, Reward={rewards['player']:.2f}")

        print("\n✅ Demo completed")

### Test Functions\n

In [11]:
# 完整训练
#main(mode='full_train')

# 5. 加载已训练模型并游玩
#main(mode='play')

### play

In [12]:
main(mode='play')

🏎️ AI Racing Game - Multi-Agent Reinforcement Learning

📂 Found local model: best_model_ep540
Using device: cuda:0
Using device: cuda:0
Using device: cuda:0
✅ Model loaded successfully

🎮 Starting Pygame UI...

Map1完成！切换到Map2...
Game over - Steps: 4784, Avg AIReward: 7.14

=== Scoreboard ===
Rank1: Player (You) - Total Score: 38462.01
Rank2: AI Car1 - Total Score: 35871.18
Rank3: AI Car2 - Total Score: 35108.22
Rank4: AI Car0 - Total Score: 31474.56
